In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("Day1").getOrCreate()

data = [
    (1, "Aarav",  "Engineering", 75000, 29, "Bangalore"),
    (2, "Priya",  "Engineering", 68000, 25, "Hyderabad"),
    (3, "Rahul",  "Sales",       55000, 31, "Mumbai"),
    (4, "Sneha",  "Engineering", 92000, 34, "Hyderabad"),
    (5, "Vikram", "HR",          48000, 28, "Delhi"),
    (6, "Anjali", "Sales",       61000, 26, "Mumbai"),
    (7, "Karan",  "Engineering", 71000, 30, "Bangalore"),
    (8, "Divya",  "HR",          52000, 24, "Delhi"),
]

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
])

df = spark.createDataFrame(data, schema)
df.show()

Question: Write PySpark code to find all employees in the Engineering department earning more than 70000, returning only name and salary, sorted by salary descending.

In [ ]:
df.filter((df.department == 'Engineering') & (df.salary > 70000)) \
  .select('name', 'salary') \
  .orderBy('salary', ascending=False) \
  .show() 

Day 2 — Aggregations

Question: For each category, find the total sales amount, the average sale amount, and the number of sales — but only show categories where total sales exceed 2000, sorted by total sales descending.

In [ ]:
data = [
    (1, "Aarav",  "Electronics", 1200, "2024-01-05"),
    (2, "Priya",  "Clothing",     450, "2024-01-06"),
    (3, "Rahul",  "Electronics",  800, "2024-01-07"),
    (4, "Sneha",  "Clothing",     300, "2024-01-08"),
    (5, "Aarav",  "Electronics", 1500, "2024-01-10"),
    (6, "Priya",  "Electronics",  700, "2024-01-11"),
    (7, "Rahul",  "Clothing",     250, "2024-01-12"),
    (8, "Sneha",  "Electronics", 1100, "2024-01-13"),
]

schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("salesperson", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("sale_date", StringType(), True),
])

df_sales = spark.createDataFrame(data, schema)

In [ ]:

df_sales.groupBy('category').count().show()

In [ ]:
from pyspark.sql.functions import count ,sum ,avg
df_sales.groupBy('category')  \
        .agg(
                sum(df_sales.amount).alias('total').cast('int'),
                avg(df_sales.amount).alias('avg'),
                count('*').alias("count") 
            ) \
        .filter('total > 2000') \
        .orderBy('total' , ascending = False) \
        .show()


# Day 3 — Joins

Dataset (orders + customers):


Question: Find all customers who placed at least one order, showing cust_name, city, and their total order amount. Then separately, identify which cust_id in orders has no matching customer record (orphaned orders). Use the appropriate join types for each.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Day1").getOrCreate()



customers_data = [
    (101, "Aarav",  "Bangalore"),
    (102, "Priya",  "Hyderabad"),
    (103, "Rahul",  "Mumbai"),
    (104, "Sneha",  "Hyderabad"),
    (105, "Vikram", "Delhi"),
]
customers_schema = StructType([
    StructField("cust_id", IntegerType(), True),
    StructField("cust_name", StringType(), True),
    StructField("city", StringType(), True),
])
df_customers = spark.createDataFrame(customers_data, customers_schema)

orders_data = [
    (1, 101, 1200),
    (2, 102, 450),
    (3, 103, 800),
    (4, 106, 300),   # cust_id 106 doesn't exist in customers
    (5, 101, 1500),
]
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("cust_id", IntegerType(), True),
    StructField("amount", IntegerType(), True),
])
df_orders = spark.createDataFrame(orders_data, orders_schema)

Answer:

Question asked
1. |cust_name|     city|total_amount|   -- get 
2. customer not match with any order



df1 = df_customers.join(df_orders, df_customers.cust_id == df_orders.cust_id, "left") \
                    .filter('order_id > 0')
df1.show()

In [ ]:
from pyspark.sql.functions import sum

df1             =   df_customers.join(df_orders, on='cust_id', how='inner') \
                   .groupBy('cust_name', 'city') \
                   .agg(sum('amount').alias('total_amount'))
df1.show()


# left_anti -- left_anti keeps only rows from the left table (df_orders) that have no match in the right table (df_customers) — and it drops all columns from the right side entirely, so you just get the orphaned order rows.

# Order that was not matched with orderdata 
# customer didnt order
orphans         = df_orders.join(df_customers, on='cust_id', how='left_anti')
orphans.show()

#  left_anti

Day 4 — Window Functions

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Day1").getOrCreate()


In [ ]:
data = [
    (1, "Aarav",  "Engineering", 75000),
    (2, "Priya",  "Engineering", 68000),
    (3, "Rahul",  "Sales",       55000),
    (4, "Sneha",  "Engineering", 92000),
    (5, "Vikram", "HR",          48000),
    (6, "Anjali", "Sales",       61000),
    (7, "Karan",  "Engineering", 71000),
    (8, "Divya",  "HR",          52000),
]
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
])
df_emp = spark.createDataFrame(data, schema)


'''
Question: For each department, rank employees by salary (highest = rank 1) using dense_rank. 
Also add a column showing the gap between each employee's salary and the highest salary in their department. 
Display name, department, salary, rank, and gap_from_top.
'''

In [ ]:
from pyspark.sql.functions import col, dense_rank, lag, lead, max as max_
from pyspark.sql.window    import Window

window_rank = Window.partitionBy('department').orderBy(col('salary').desc())
window_dept = Window.partitionBy('department')  # no orderBy -> covers full partition

result = df_emp.withColumn("rank", dense_rank().over(window_rank)) \
               .withColumn("highest", max_("salary").over(window_dept)) \
               .withColumn("gap_from_top", col("highest") - col("salary")) \
               .select("name", "department", "salary", "rank", "gap_from_top")
result.show()

Day 5 — Strings, Dates, UDFs

```text
Question:

1.Convert order_date and ship_date from string to actual date type, and add a days_to_ship column (difference in days).
2.Properly capitalize customer_name (e.g. "aarav sharma" → "Aarav Sharma").
3.Extract the category prefix from product_code (e.g. "ELEC" from "ELEC-1234") into a new column category.
4.Write a UDF that labels each order "Delayed" if days_to_ship > 5, else "On Time".
```

In [ ]:
import os
import sys
os.environ['HADOOP_HOME'] = 'C:\\hadoop'


os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Day1").getOrCreate()

data = [
    (1, "aarav sharma", "2024-01-05", "2024-01-09", "ELEC-1234"),
    (2, "priya patel",  "2024-02-10", "2024-02-15", "CLTH-5678"),
    (3, "rahul singh",  "2024-03-21", "2024-03-22", "ELEC-9012"),
    (4, "sneha gupta",  "2024-04-02", "2024-04-10", "HOME-3456"),
]
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("ship_date", StringType(), True),
    StructField("product_code", StringType(), True),
])
df_orders2 = spark.createDataFrame(data, schema)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date,date_diff
# date_diff - days differents
# months_between - months differents


# Date
df1 = df_orders2.withColumn("order_date", to_date("order_date", "yyyy-MM-dd")) \
                .withColumn("ship_date", to_date("ship_date", "yyyy-MM-dd"))  

df2 = df1.withColumn('days_to_ship' , datediff(df1['ship_date'] ,df1['order_date']))            
df2.show()
df2.printSchema()


In [ ]:
#  from pyspark.sql.functions import initcap

from pyspark.sql.functions import initcap ,split


df = df_orders2.withColumn("customer_name", initcap("customer_name")) \
                .withColumn("product_code_new", split("product_code","-")[0])

df.show()

In [ ]:

from pyspark.sql            import SparkSession
from pyspark.sql.functions  import *
from pyspark.sql.types      import *



df1 = df_orders2.withColumn("order_date", to_date("order_date", "yyyy-MM-dd")) \
                .withColumn("ship_date", to_date("ship_date", "yyyy-MM-dd"))  

df2 = df1.withColumn('days_to_ship' , datediff(df1['ship_date'] ,df1['order_date']))    \
            .withColumn("product_code_new", split("product_code","-")[0])        
df2.show()
df2.printSchema()

@udf(StringType())
def OrderStatus(days_to_ships):
    if days_to_ships is None:
        return None
    elif days_to_ships > 5:
        return "Delayed"
    else:
        return "On Time"



df3 = df2.withColumn('status', OrderStatus(col(("days_to_ship"))))
df3.show()

'''
#without udf
# 
from pyspark.sql.functions import when, col

df3 = df2.withColumn(
    "status",
    when(col("days_to_ship") > 5, "Delayed").otherwise("On Time")
)

'''








Interviewers frequently probe "would you use a UDF here?" specifically to see if you reach for native functions first and only fall back to UDFs (or better, pandas UDFs) when the logic truly can't be expressed natively. You wrote a correct UDF, but flagging when not to use one is the senior-level answer.

...completed correctely Till here udf not working in Local Machine

butt working in Pyspark - databricks

In [ ]:
# Day 6 — Performance Tuning


In [2]:
from pyspark.sql            import SparkSession
from pyspark.sql.functions  import *
from pyspark.sql.types      import *


spark = SparkSession.builder.appName('day_06').getOrCreate()

In [3]:
# Large fact table (imagine millions of rows in reality)
orders_data = [(i, i % 5 + 1, 100 + i) for i in range(1, 21)]
df_big_orders = spark.createDataFrame(orders_data, ["order_id", "category_id", "amount"])

# Small lookup table
categories_data = [(1, "Electronics"), (2, "Clothing"), (3, "Home"), (4, "Books"), (5, "Toys")]
df_categories = spark.createDataFrame(categories_data, ["category_id", "category_name"])

In [7]:
print(df_big_orders.count())
print(df_categories.count())

20
5


In [6]:
from pyspark.sql.functions import broadcast, col

orders_data = [(i, i % 5 + 1, 100 + i) for i in range(1, 21)]
df_big_orders = spark.createDataFrame(orders_data, ["order_id", "category_id", "amount"])

categories_data = [(1, "Electronics"), (2, "Clothing"), (3, "Home"), (4, "Books"), (5, "Toys")]
df_categories = spark.createDataFrame(categories_data, ["category_id", "category_name"])

# Part 1: small lookup table -> broadcast it, so the large table is never shuffled
df_joined = df_big_orders.join(broadcast(df_categories), on="category_id", how="inner")

# Part 2: reused 3x downstream -> cache so the join isn't recomputed each time
df_joined.cache()
df_joined.count()  # forces Spark to materialize the cache right now

# use 1 — aggregation
df_summary = df_joined.groupBy("category_name").sum("amount")
df_summary.show()

# use 2 — filter
df_high_value = df_joined.filter(col("amount") > 110)
df_high_value.show()

# use 3 — write
# Part 3: 200 partitions worth of data fits comfortably in 10 -> coalesce
# (cheap, no shuffle, just merges adjacent partitions; repartition would
# shuffle everything for the sake of perfectly even sizes, which isn't needed here)
df_joined.coalesce(10).write.mode("overwrite").parquet("output_path")

df_joined.unpersist()  # release the cached memory once done

+-------------+-----------+
|category_name|sum(amount)|
+-------------+-----------+
|     Clothing|        434|
|         Home|        438|
|        Books|        442|
|         Toys|        446|
|  Electronics|        450|
+-------------+-----------+

+-----------+--------+------+-------------+
|category_id|order_id|amount|category_name|
+-----------+--------+------+-------------+
|          2|      11|   111|     Clothing|
|          3|      12|   112|         Home|
|          4|      13|   113|        Books|
|          5|      14|   114|         Toys|
|          1|      15|   115|  Electronics|
|          2|      16|   116|     Clothing|
|          3|      17|   117|         Home|
|          4|      18|   118|        Books|
|          5|      19|   119|         Toys|
|          1|      20|   120|  Electronics|
+-----------+--------+------+-------------+



Py4JJavaError: An error occurred while calling o139.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

Q8.
Scenario: You're building a monthly revenue report. For each customer, each month, you need total revenue, their rank against other customers that month, and a "VIP" flag for the top 3 spenders of each month.

In [9]:
from pyspark.sql.functions import to_date

customers_data = [
    (1, "Aarav", "Bangalore"),
    (2, "Priya", "Hyderabad"),
    (3, "Rahul", "Mumbai"),
    (4, "Sneha", "Hyderabad"),
    (5, "Vikram", "Delhi"),
]
df_customers = spark.createDataFrame(customers_data, ["cust_id", "name", "city"])

products_data = [
    (101, "Laptop", "Electronics", 50000),
    (102, "T-Shirt", "Clothing", 500),
    (103, "Phone", "Electronics", 30000),
    (104, "Sofa", "Home", 20000),
]
df_products = spark.createDataFrame(products_data, ["product_id", "product_name", "category", "price"])

orders_data = [
    (1, 1, 101, 1, "2024-01-05"),
    (2, 2, 102, 3, "2024-01-06"),
    (3, 3, 103, 1, "2024-01-07"),
    (4, 1, 102, 2, "2024-01-15"),
    (5, 4, 101, 1, "2024-02-02"),
    (6, 5, 104, 1, "2024-02-03"),
    (7, 1, 103, 1, "2024-02-10"),
    (8, 2, 101, 1, "2024-02-12"),
    (9, 3, 102, 5, "2024-02-15"),
    (10, 4, 104, 1, "2024-02-20"),
]
df_orders = spark.createDataFrame(orders_data, ["order_id", "cust_id", "product_id", "quantity", "order_date"])
df_orders = df_orders.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))

In [10]:
from pyspark.sql.functions import broadcast, col, sum as _sum, date_format, dense_rank, when
from pyspark.sql.window import Window

# Step 1: customers and products are tiny lookup tables -> broadcast join,
# avoids shuffling the (in reality, much larger) orders table
df_joined = df_orders.join(broadcast(df_customers), on="cust_id", how="inner") \
                      .join(broadcast(df_products), on="product_id", how="inner")

# Step 2: derive revenue and a year-month key
df_enriched = df_joined.withColumn("revenue", col("quantity") * col("price")) \
                        .withColumn("month", date_format("order_date", "yyyy-MM"))

# Step 3: aggregate revenue per customer per month
df_monthly = df_enriched.groupBy("month", "cust_id", "name") \
                         .agg(_sum("revenue").alias("total_revenue"))

# Cache here: this DataFrame feeds both the ranking step AND the final write,
# so without caching, Spark would redo the join+aggregation twice
df_monthly.cache()

# Step 4: rank customers within each month by revenue
window_month = Window.partitionBy("month").orderBy(col("total_revenue").desc())
df_ranked = df_monthly.withColumn("rank", dense_rank().over(window_month))

# Step 5: flag VIPs — when/otherwise instead of a UDF, since this is simple
# conditional logic that native Spark functions handle natively and faster
df_final = df_ranked.withColumn("status", when(col("rank") <= 3, "VIP").otherwise("Regular"))

df_final.orderBy("month", "rank").show()

# Step 6: write partitioned by month; coalesce first since the real output
# would otherwise be split across far more partitions than the data needs
df_final.coalesce(5).write.partitionBy("month").mode("overwrite").parquet("monthly_report")

df_monthly.unpersist()

+-------+-------+------+-------------+----+-------+
|  month|cust_id|  name|total_revenue|rank| status|
+-------+-------+------+-------------+----+-------+
|2024-01|      1| Aarav|        51000|   1|    VIP|
|2024-01|      3| Rahul|        30000|   2|    VIP|
|2024-01|      2| Priya|         1500|   3|    VIP|
|2024-02|      4| Sneha|        70000|   1|    VIP|
|2024-02|      2| Priya|        50000|   2|    VIP|
|2024-02|      1| Aarav|        30000|   3|    VIP|
|2024-02|      5|Vikram|        20000|   4|Regular|
|2024-02|      3| Rahul|         2500|   5|Regular|
+-------+-------+------+-------------+----+-------+



Py4JJavaError: An error occurred while calling o409.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

explaination:

Spark: nothing actually runs until you call an action like .show(), .count(), or .write()

.cache() is also lazy — it just marks the DataFrame as "save your output in memory once computed." It doesn't do anything by itself. That's why the very next line is .count() — calling count is a trick to force Spark to actually run the join now, while it's caching, rather than waiting for the first real action later. After this line runs, the joined result genuinely sits in memory.






Part 1 — the join
pythondf_joined = df_big_orders.join(broadcast(df_categories), on="category_id", how="inner")
Without broadcast(), Spark would normally shuffle both tables — meaning it redistributes rows across the cluster so that matching category_id values land on the same machine, then joins them there. For a huge orders table, that's a lot of network traffic and disk I/O just to line things up.
broadcast(df_categories) tells Spark "this one's small enough — just copy the whole thing to every machine in the cluster." Now each machine already has the full categories table sitting in memory, so it can match each order to its category name locally, with zero shuffling of the big table. That's the entire performance win.
Part 2 — caching
pythondf_joined.cache()
df_joined.count()
Here's the thing about Spark: nothing actually runs until you call an action like .show(), .count(), or .write(). Up to this point, df_joined is just a plan — Spark hasn't actually executed the join yet.
.cache() is also lazy — it just marks the DataFrame as "save your output in memory once computed." It doesn't do anything by itself. That's why the very next line is .count() — calling count is a trick to force Spark to actually run the join now, while it's caching, rather than waiting for the first real action later. After this line runs, the joined result genuinely sits in memory.
Why bother? Because below this, df_joined gets used three separate times. Without caching, each of those three uses would trigger Spark to redo the entire join from scratch — three times. With caching, the join happens once, and all three downstream steps just read the already-computed result from memory.
The three downstream uses
pythondf_summary = df_joined.groupBy("category_name").sum("amount")
df_summary.show()
Groups the cached result by category name and sums the amounts — gives you total revenue per category. .show() is the action that triggers it; since df_joined is cached, this just reads from memory rather than rejoining.
pythondf_high_value = df_joined.filter(col("amount") > 110)
df_high_value.show()
A completely separate use of the same cached DataFrame — filters down to only orders above 110. Again, no rejoin happens because it's reading the cached version.
pythondf_joined.coalesce(10).write.mode("overwrite").parquet("output_path")
This is the third use — writing the data out. But before writing, coalesce(10) matters: if df_joined happened to be spread across, say, 200 partitions internally (common after shuffles or with large source data), writing it directly would produce 200 separate small files on disk — bad for storage efficiency and bad for whoever reads it back later. coalesce(10) merges those partitions down to 10 without triggering a shuffle (it just glues neighboring partitions together), so you get 10 reasonably-sized files instead of 200 tiny ones.
Cleanup
pythondf_joined.unpersist()
Once you're done reusing the cached DataFrame, this tells Spark to free up the memory it was holding. Skipping this isn't fatal, but on a long-running job with many cached DataFrames, forgetting to unpersist is a common way to slowly eat up cluster memory.
Net effect of the whole script: one join (done efficiently via broadcast), computed once (via caching) and reused three ways, then written out efficiently (via coalesce) — that's the "good Spark hygiene" combination interviewers are listening for when they ask performance questions.



